# ODI to Databricks Migration: `TRG_EMP.sql`

**Conversion Timestamp:** 2024-07-30

**Description:** This notebook loads data into the `TRG_EMP` target table from the `EMPLOYEES` source table.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "FULL", "ETL Job Type (FULL/INCREMENTAL)");
dbutils.widgets.text("DATASOURCE_NUM_ID", "0", "Datasource Number ID");
dbutils.widgets.text("ETL_PROC_WID", "0", "ETL Process Widget ID");
dbutils.widgets.text("ODI_SESS_NO", "0", "ODI Session Number");
dbutils.widgets.text("ETL_LAST_EXTRACT_TIME", "1900-01-01 00:00:00", "Last Extract Time");
dbutils.widgets.text("ETL_CURRENT_EXTRACT_TIME", "1900-01-01 00:00:00", "Current Extract Time");

# ETL Parameters

These parameters control the ETL process and are passed in from the calling workflow or set with default values.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_parameters AS
SELECT
  '${ETL_JOB_TYPE}' AS etl_job_type,
  CAST('${DATASOURCE_NUM_ID}' AS BIGINT) AS datasource_num_id,
  CAST('${ETL_PROC_WID}' AS BIGINT) AS etl_proc_wid,
  CAST('${ODI_SESS_NO}' AS BIGINT) AS odi_sess_no,
  to_timestamp('${ETL_LAST_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_last_extract_time,
  to_timestamp('${ETL_CURRENT_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_current_extract_time;

In [ ]:
display(spark.sql("SELECT * FROM v_etl_parameters"))

# Target Table Load

In [ ]:
%sql
-- SCEN_TASK_NO {10}
-- SCEN_TASK_NO {20}
-- SCEN_TASK_NO {30}: Insert into target
INSERT INTO workspace.hr.trg_emp
  (
    employee_id ,
    first_name ,
    last_name ,
    email ,
    phone_number ,
    hire_date ,
    job_id ,
    salary ,
    commission_pct ,
    manager_id ,
    department_id 
  ) 
SELECT 
  employees.employee_id ,
  employees.first_name ,
  employees.last_name ,
  employees.email ,
  employees.phone_number ,
  employees.hire_date ,
  employees.job_id ,
  employees.salary ,
  employees.commission_pct ,
  employees.manager_id ,
  employees.department_id  
FROM 
  workspace.hr.employees AS employees;

# Validation

In [ ]:
%sql
SELECT COUNT(*) AS total_records FROM workspace.hr.trg_emp;

# Conversion Notes

- The Oracle hint `/*+ APPEND PARALLEL */` was removed as it is not applicable to Databricks Delta Lake.
- Schema `HR` was converted to `workspace.hr` and table names `TRG_EMP`, `EMPLOYEES` were lowercased to `trg_emp`, `employees` following naming conventions.
- The original SQL was a simple `INSERT INTO ... SELECT FROM`. No `MERGE` or complex logic was detected or implemented.
- Placeholder Databricks widgets and temporary views for common ETL parameters were included for structural completeness, although not directly referenced in this specific SQL statement.